# Confronto di due punti senza scansione

Acquisisce lo **stesso segnale (CH2) in due punti** (`PUNTO_1`, `PUNTO_2`), **senza scansione del laser** (laser fermo alla sua lunghezza d'onda, scope in single con trigger forzato), poi confronta i due segnali e ne calcola il **rapporto** `PUNTO_2 / PUNTO_1`.

- Le acquisizioni sono **alternate** (P1, P2, P1, P2, ...) per `N_ACQUISIZIONI` giri, così una deriva lenta di potenza o di scope pesa uguale sui due punti.
- Il livello di ciascun punto è la **mediana di CH2** (i picchi spuri non contano); l'incertezza viene dalla dispersione tra i giri (o, con un solo giro, dai blocchi del record).
- Si leggono solo i primi `N_CAMPIONI` campioni del record (senza scansione il segnale è stazionario: non serve leggere tutto).

Ordine: celle 1-3 una volta per sessione (la cella 1 contiene tutte le impostazioni); la cella 4 mostra l'anteprima (**non muove niente**); la cella 5 acquisisce (chiede conferma); la cella 6 fa i grafici e il rapporto, e si può rilanciare da sola su una cartella già acquisita (`CARTELLA_DATI`).


In [ ]:
# Cella 1 - configurazione e funzioni
import csv
import os
import time
import datetime

import numpy as np
import pyvisa

OSC_VISA = "USB::0x0699::0x0408::C028813::INSTR"
OSC_TIMEOUT_MS = 30000
CHANNELS = ["CH1", "CH2"]      # canali salvati; il confronto usa CH2

# ---------------- i due punti (in mm, limiti motori 0-150) ----------------
PUNTO_1 = None             # <-- SCRIVI QUI (X, Y) del primo punto, es. (91.50, 146.20)
PUNTO_2 = None             # <-- SCRIVI QUI (X, Y) del secondo punto
N_ACQUISIZIONI = 3         # giri: ogni giro acquisisce P1 e poi P2
N_CAMPIONI = 100000        # campioni letti per acquisizione (None = tutto il record: file da ~240 MB)
ASSESTAMENTO_S = 1.0       # pausa dopo il movimento, prima di acquisire
POLL_INTERVAL_S = 0.1
CHANNEL_READ_PAUSE_S = 0.1
# --------------------------------------------------------------------------


def configura_scope(osc):
    osc.write("HORizontal:MODE MANUAL")
    rec_len = int(osc.query("HORizontal:RECORDLENGTH?"))
    osc.write("DATA:START 1")
    osc.write(f"DATA:STOP {rec_len if N_CAMPIONI is None else min(rec_len, int(N_CAMPIONI))}")
    osc.write("DATA:WIDTH 1")
    osc.write("DATA:ENCdg ASCii")


def arm_acquisition(osc, timeout_s=120):
    osc.write("ACQ:STOPAFTER SEQUENCE")
    osc.write("ACQ:STATE RUN")
    t0 = time.time()
    while osc.query("TRIGger:STATE?").strip().upper() != "READY":
        if time.time() - t0 > timeout_s:
            raise TimeoutError(f"Lo scope non e' arrivato a READY in {timeout_s} s")
        time.sleep(POLL_INTERVAL_S)


def wait_for_acquisition(osc):
    while int(osc.query("ACQ:STATE?")) != 0:
        time.sleep(POLL_INTERVAL_S)


def acquire_channel(osc, source="CH1"):
    osc.write(f"DATA:SOURCE {source}")
    ymult = float(osc.query("WFMPRE:YMULT?"))
    yoff = float(osc.query("WFMPRE:YOFF?"))
    yzero = float(osc.query("WFMPRE:YZERO?"))
    ydata = np.array(osc.query("CURVE?").split(","), dtype=float)
    return (ydata - yoff) * ymult + yzero


In [ ]:
# Cella 2 - connessione all'oscilloscopio (una volta per sessione)
rm = pyvisa.ResourceManager()
osc = rm.open_resource(OSC_VISA)
osc.timeout = OSC_TIMEOUT_MS
configura_scope(osc)
print("Oscilloscopio connesso e configurato.")

In [ ]:
# Cella 3 - motori PI C-863: X = ASRL4, Y = ASRL5 (una volta per sessione)
MOTORE_X = "ASRL4::INSTR"
MOTORE_Y = "ASRL5::INSTR"
PI_BAUD = 19200
PI_AXIS = "1"
PI_TIMEOUT_MS = 2000
PI_TIMEOUT_MOVIMENTO_S = 60
PI_TOLLERANZA_MM = 0.01  # valore usato nella prova sul laboratorio (i motori arrivano entro circa 10 um)


def pi_apri(porta):
    inst = rm.open_resource(porta)
    inst.baud_rate = PI_BAUD
    inst.timeout = PI_TIMEOUT_MS
    inst.read_termination = "\n"
    inst.write_termination = "\n"
    return inst


def pi_query(inst, comando):
    return inst.query(comando).strip()


def pi_valore(inst, comando):
    """Estrae il numero da risposte tipo '1=+0012.3456'."""
    return float(pi_query(inst, comando).split("=")[-1])


def pi_limiti(inst):
    return pi_valore(inst, f"TMN? {PI_AXIS}"), pi_valore(inst, f"TMX? {PI_AXIS}")


def pi_muovi(inst, nome, target):
    """Movimento assoluto in mm, con controllo dei limiti prima di muovere."""
    tmin, tmax = pi_limiti(inst)
    if not (tmin <= target <= tmax):
        raise ValueError(f"Motore {nome}: target {target} mm fuori dai limiti {tmin}..{tmax} mm")
    inst.write(f"MOV {PI_AXIS} {target:.4f}")
    time.sleep(0.1)
    t0 = time.time()
    while True:
        in_target = pi_query(inst, f"ONT? {PI_AXIS}").endswith("1")
        pos = pi_valore(inst, f"POS? {PI_AXIS}")
        if in_target and abs(pos - target) <= PI_TOLLERANZA_MM:
            break
        if time.time() - t0 > PI_TIMEOUT_MOVIMENTO_S:
            raise TimeoutError(f"Motore {nome}: non arrivato a {target} mm (posizione {pos} mm)")
        time.sleep(0.1)
    errore = int(pi_query(inst, "ERR?"))
    if errore != 0:
        raise RuntimeError(f"Motore {nome}: ERR? = {errore} dopo il movimento a {target} mm")
    print(f"Motore {nome}: a {pos:.4f} mm")


motore_x = pi_apri(MOTORE_X)
motore_y = pi_apri(MOTORE_Y)

for nome, inst in (("X", motore_x), ("Y", motore_y)):
    errore_residuo = pi_query(inst, "ERR?")  # legge e azzera un eventuale errore vecchio
    tmin, tmax = pi_limiti(inst)
    print(f"Motore {nome}: {pi_query(inst, '*IDN?')}")
    print(f"   posizione {pi_valore(inst, f'POS? {PI_AXIS}')} mm | limiti {tmin}..{tmax} mm | "
          f"servo {pi_query(inst, f'SVO? {PI_AXIS}')} | referenziato {pi_query(inst, f'FRF? {PI_AXIS}')} | "
          f"errore residuo letto: {errore_residuo}")

In [ ]:
# Cella 4 - ANTEPRIMA (NON muove niente)
if PUNTO_1 is None or PUNTO_2 is None:
    raise ValueError("Scrivi in cella 1 le coordinate (X, Y) di PUNTO_1 e PUNTO_2 e rilancia la cella 1")

PUNTI = [(float(PUNTO_1[0]), float(PUNTO_1[1])), (float(PUNTO_2[0]), float(PUNTO_2[1]))]
for nome, inst, k in (("X", motore_x, 0), ("Y", motore_y, 1)):
    tmin, tmax = pi_limiti(inst)
    for n, p in enumerate(PUNTI, start=1):
        if not (tmin <= p[k] <= tmax):
            raise ValueError(f"Punto {n}: {nome} = {p[k]} mm fuori dai limiti {tmin}..{tmax} mm")

for n, (x, y) in enumerate(PUNTI, start=1):
    print(f"  punto {n}: X={x:.4f}  Y={y:.4f}")
print(f"{int(N_ACQUISIZIONI)} giri x 2 punti = {2 * int(N_ACQUISIZIONI)} acquisizioni, {'tutto il record' if N_CAMPIONI is None else str(int(N_CAMPIONI)) + ' campioni'} ciascuna, senza scansione (laser fermo)")


In [ ]:
# Cella 5 - ACQUISISCE: a ogni giro muove su P1 e acquisisce, poi su P2 e acquisisce (senza scansione)
if input(f"Muovere i motori e acquisire {2 * int(N_ACQUISIZIONI)} segnali? scrivi 'si' per confermare: ").strip().lower() != "si":
    raise RuntimeError("Annullato: nessun movimento eseguito.")

oggi = datetime.date.today().strftime("%Y-%m-%d")
base_folder = os.path.join(os.getcwd(), oggi)
os.makedirs(base_folder, exist_ok=True)
esistenti = [d for d in os.listdir(base_folder) if d.startswith("conf_")]
conf_folder = os.path.join(base_folder, f"conf_{len(esistenti) + 1:03d}")
os.makedirs(conf_folder)
print(f"Dati in: {conf_folder}")

registro = os.path.join(conf_folder, "acquisizioni.csv")
with open(registro, "w", newline="") as f:
    csv.writer(f).writerow(["indice", "punto", "giro", "x_mm", "y_mm", "file"])

contatore = 0
for giro in range(int(N_ACQUISIZIONI)):
    for n, (x, y) in enumerate(PUNTI, start=1):
        print(f"\n=== giro {giro + 1}/{int(N_ACQUISIZIONI)}, punto {n}: X={x:.4f}, Y={y:.4f} ===")
        pi_muovi(motore_x, "X", x)
        pi_muovi(motore_y, "Y", y)
        time.sleep(ASSESTAMENTO_S)

        arm_acquisition(osc)
        osc.write("TRIGger FORCe")                # nessuna scansione: trigger forzato
        wait_for_acquisition(osc)
        time.sleep(1.0)

        data = {}
        for ch in CHANNELS:
            data[ch] = acquire_channel(osc, ch)
            time.sleep(CHANNEL_READ_PAUSE_S)
        nome_file = f"punto{n}_giro{giro + 1}.csv"
        min_len = min(len(v) for v in data.values())
        colonne = np.column_stack([data[ch][:min_len] for ch in CHANNELS])
        np.savetxt(os.path.join(conf_folder, nome_file), colonne, delimiter=",", header=",".join(CHANNELS), comments="")
        with open(registro, "a", newline="") as f:
            csv.writer(f).writerow([contatore, n, giro + 1, f"{x:.4f}", f"{y:.4f}", nome_file])
        contatore += 1
        print(f"Salvato: {nome_file} (mediana CH2 = {np.median(data['CH2'][:min_len]):.5g} V)")

print(f"\nAcquisiti {contatore} file in {conf_folder}")
print("Per rifare l'analisi su questa cartella, in cella 6:  CARTELLA_DATI = " + repr(os.path.relpath(conf_folder, os.getcwd()).replace(os.sep, "/")))


In [ ]:
# Cella 6 - confronto dei due punti e rapporto (anche su una cartella gia' acquisita)
import matplotlib.pyplot as plt

CARTELLA_DATI = ""         # vuoto = la cartella appena acquisita in cella 5; altrimenti es. "2026-10-15/conf_001"
CANALE = "CH2"
LIVELLO_BUIO_V = 0.0       # livello di CH2 senza luce, sottratto da entrambi i punti
BLOCCO = 1000              # campioni per blocco (mediana) nei grafici dei segnali

cartella = os.path.abspath(CARTELLA_DATI) if CARTELLA_DATI.strip() else conf_folder
with open(os.path.join(cartella, "acquisizioni.csv"), newline="") as f:
    righe = list(csv.DictReader(f))


def carica(file):
    ch2 = np.loadtxt(os.path.join(cartella, file), delimiter=",", skiprows=1, usecols=CHANNELS.index(CANALE) if CANALE in CHANNELS else 1)
    return ch2 - LIVELLO_BUIO_V


def a_blocchi(s):
    k = len(s) // BLOCCO
    return np.median(s[:k * BLOCCO].reshape(k, BLOCCO), axis=1)


segnali = {1: [], 2: []}                  # punto -> lista di (blocchi, livello, errore_se_singolo)
pos = {}
for r in righe:
    s = carica(r["file"])
    b = a_blocchi(s)
    segnali[int(r["punto"])].append((b, float(np.median(s)), float(b.std(ddof=1) / np.sqrt(len(b))) if len(b) > 1 else 0.0))
    pos[int(r["punto"])] = (r["x_mm"], r["y_mm"])


def livello(lista):
    liv = np.array([l for _, l, _ in lista])
    if len(liv) > 1:
        return liv.mean(), liv.std(ddof=1) / np.sqrt(len(liv))      # dispersione tra i giri
    return liv[0], lista[0][2]                                        # un solo giro: dai blocchi del record


L1, e1 = livello(segnali[1])
L2, e2 = livello(segnali[2])
R = L2 / L1
eR = abs(R) * np.sqrt((e1 / L1) ** 2 + (e2 / L2) ** 2)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for p, colore in ((1, "tab:blue"), (2, "tab:red")):
    for k, (b, _, _) in enumerate(segnali[p]):
        a1.plot(np.arange(len(b)) * BLOCCO, b, color=colore, lw=0.8, alpha=0.7, label=f"punto {p} (X={pos[p][0]}, Y={pos[p][1]})" if k == 0 else None)
a1.axhline(L1, color="tab:blue", ls="--", lw=1)
a1.axhline(L2, color="tab:red", ls="--", lw=1)
a1.set_ylabel(f"{CANALE} (V)")
a1.set_title("Segnale nei due punti (mediana a blocchi; tratteggio = livello)")
a1.legend(loc="best")

for k in range(min(len(segnali[1]), len(segnali[2]))):
    b1, b2 = segnali[1][k][0], segnali[2][k][0]
    n = min(len(b1), len(b2))
    a2.plot(np.arange(n) * BLOCCO, b2[:n] / b1[:n], color="k", lw=0.8, alpha=0.7, label="punto 2 / punto 1 (per giro)" if k == 0 else None)
a2.axhline(R, color="tab:green", lw=2, label=f"rapporto dei livelli = {R:.4f} ± {eR:.4f}")
a2.axhline(1, color="gray", lw=0.8, ls=":")
a2.set_ylabel("punto 2 / punto 1")
a2.set_xlabel("campione")
a2.legend(loc="best")
plt.tight_layout()
plt.show()

print(f"Punto 1 (X={pos[1][0]}, Y={pos[1][1]}): {L1:.5g} ± {e1:.2g} V  ({len(segnali[1])} acquisizioni)")
print(f"Punto 2 (X={pos[2][0]}, Y={pos[2][1]}): {L2:.5g} ± {e2:.2g} V  ({len(segnali[2])} acquisizioni)")
print(f"Rapporto punto 2 / punto 1 = {R:.5f} ± {eR:.5f}   (punto 1 / punto 2 = {1 / R:.5f})")
